# Layer 3 — Phase 5 v3: 2x2 책임 매트릭스 시나리오 시뮬레이션

## v2 → v3 변경사항 (6개)

### 필수 (계산 정확성)
1. **★ 케이스 분류 기준: `incident_flag` → `L3_trigger`** — 보상은 trigger 발동 기준
2. **★ `n_days = unique date` 개수** — 불연속 일자에서도 정확한 연간 환산

### 권장
3. **`infra_actual` / `infra_trigger` 분리** — 가독성 + 명확성
4. **"L2 독립적" → "민감하지 않음"** 표현 완화
5. **C/D placeholder 금액 발표 표에서 숨김**
6. **Synthetic scenario 강조 metadata 추가**

## v2 핵심 버그 (수정됨)
- v2 잘못된 흐름:
  ```
  infra_fault = incident_flag  # 실제 장애
  → Case B 분류 → L3 보상 계산
  문제: trigger 안 발동했는데 보상 계산함
  ```
- v3 수정:
  ```
  infra_trigger = L3_trigger    # 보험 trigger 발동
  → Case B 분류 → L3 보상 계산
  의미: trigger 발동 기준 보상이 보험 도메인 정합
  ```

## 2x2 책임 매트릭스 (v3 재정의)
```
                    AI 정탐 성공       AI 정탐 실패
                  ┌───────────────┬───────────────┐
L3 trigger X     │  A: 정상 운영  │  C: L2 단독    │
                  │  (보상 없음)   │  (L5 결정)     │
                  ├───────────────┼───────────────┤
L3 trigger 발동  │  B: L3 단독    │  D: L2+L3 병합 │
                  │  (지수형 Y_L3) │  (L5 결정)     │
                  └───────────────┴───────────────┘
```

**Case B 정의 (v3 명확화)**:
> *"L3 trigger 발동했지만 AI 공정이 이상을 정탐하여 배치 폐기까지 가지 않은 경우. L3 지수형 보상만 지급."*

## Step A. 환경 셋업 + Part 4 결과 로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

ROOT = '/content/drive/MyDrive/layer3_data'
OUT = f'{ROOT}/processed'

pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 4)

!apt -qq install fonts-nanum > /dev/null 2>&1
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)

In [ ]:
layer3 = pd.read_parquet(os.path.join(OUT, 'layer3_final_output.parquet'))

with open(os.path.join(OUT, 'selected_trigger_config.json'), 'r', encoding='utf-8') as f:
    trigger_config = json.load(f)

Y_L3 = trigger_config['Y_payout_krw']
THETA = trigger_config['theta']
N_MIN = trigger_config['N_minutes']

print(f'layer3_final_output: {layer3.shape}')
print(f'L3 trigger: P_cloud >= {THETA:.2f} for >= {N_MIN}분')
print(f'Y_L3 = {Y_L3/1e8:.2f}억원/event')
print(f'\n실제 incident 분: {layer3["incident_flag"].sum()}')
print(f'L3 trigger 분: {layer3["L3_trigger"].sum()}')

# v3: 두 변수 차이 명시
print(f'\n=== v3 안내 ===')
print(f'incident_flag: 실제 cloud 장애 (모델 평가용)')
print(f'L3_trigger: 보험 트리거 발동 (보상 시뮬용)')

## Step B. Synthetic Layer 2 시나리오 생성 (v3: 변수 분리)

**v3 변경**:
- `infra_actual` (실제 장애) → process_fault 발생 확률 계산용
- `infra_trigger` (보험 trigger 발동) → 보상 케이스 분류용

In [ ]:
L2_DETECTION_ACCURACY = 0.70   # 보수적 가정
P_PROCESS_FAULT_GIVEN_INFRA = 0.70
P_PROCESS_FAULT_GIVEN_NORMAL = 0.05

# ★ v3: 두 변수 명확히 분리
infra_actual = (layer3['incident_flag'] == 1).values   # 실제 장애 (process fault 확률)
infra_trigger = (layer3['L3_trigger'] == 1).values      # 보험 트리거 (보상 분류)

# Process fault 발생 확률은 실제 cloud incident와 상관 있다고 가정
# (인프라가 실제로 안 좋으면 공정에도 영향)
p_process = np.where(
    infra_actual,
    P_PROCESS_FAULT_GIVEN_INFRA,
    P_PROCESS_FAULT_GIVEN_NORMAL
)
process_fault = np.random.binomial(1, p_process)

# AI decision (정확도 L2_DETECTION_ACCURACY)
p_ai_detect = np.where(
    process_fault == 1,
    L2_DETECTION_ACCURACY,
    1 - L2_DETECTION_ACCURACY
)
ai_decision = np.random.binomial(1, p_ai_detect)
ai_correct = (process_fault == ai_decision).astype(int)

layer3['process_fault'] = process_fault
layer3['ai_decision'] = ai_decision
layer3['ai_correct'] = ai_correct

print(f'=== Synthetic L2 시나리오 (v3) ===')
print(f'L2 정탐 정확도: {L2_DETECTION_ACCURACY:.2f} (보수적)')
print(f'\nProcess fault 발생: {process_fault.sum()} / {len(process_fault)}')
print(f'  - infra_actual=1일 때: {process_fault[infra_actual].sum()} / {infra_actual.sum()}')
print(f'  - infra_actual=0일 때: {process_fault[~infra_actual].sum()} / {(~infra_actual).sum()}')
print(f'\nAI 정탐 정확: {ai_correct.sum()} / {len(ai_correct)} ({ai_correct.mean():.4f})')
print(f'\nInfra_trigger 분 (보상 기준): {infra_trigger.sum()} / {len(infra_trigger)}')

## Step C. 2x2 매트릭스 분류 ⭐ v3 핵심

**v2 → v3**: `incident_flag` 기준 → **`L3_trigger` 기준**

**근거**: 보험은 trigger 발동에 따라 지급. 실제 장애가 있어도 trigger 안 발동하면 보상 없음.

In [ ]:
def classify_case_v3(row):
    """v3: L3_trigger 기준 케이스 분류 (보험 도메인 정합)"""
    l3_triggered = row['L3_trigger'] == 1
    ai_ok = row['ai_correct'] == 1
    
    if not l3_triggered and ai_ok:
        return 'A'   # 정상 운영 — L3 trigger X + AI 정탐 OK
    elif l3_triggered and ai_ok:
        return 'B'   # L3 단독 — trigger 발동했으나 AI도 정탐
    elif not l3_triggered and not ai_ok:
        return 'C'   # L2 단독 — trigger X but AI 오탐 (공정 fault만)
    else:
        return 'D'   # L2+L3 병합 — trigger 발동 + AI 오탐

layer3['case'] = layer3.apply(classify_case_v3, axis=1)

case_counts = layer3['case'].value_counts().sort_index()
case_pct = layer3['case'].value_counts(normalize=True).sort_index() * 100

summary_basic = pd.DataFrame({
    'minute_count': case_counts,
    'percent': case_pct.round(2)
})
print('=== 케이스별 분포 (v3: L3_trigger 기준) ===')
display(summary_basic)

# Sanity check: v3에서 Case B + D = L3_trigger 발동 분 수
trigger_minutes = (layer3['L3_trigger'] == 1).sum()
b_plus_d = (layer3['case'].isin(['B', 'D'])).sum()
print(f'\n검증: L3_trigger 발동 분 = {trigger_minutes}, Case B+D = {b_plus_d}')
assert trigger_minutes == b_plus_d, '케이스 분류 오류 — Case B+D ≠ trigger 발동 분'
print('✓ 검증 통과: Case B+D 합 = L3_trigger 발동 분')

## Step D. 케이스별 빈도 + 보험금

In [ ]:
# L2 관련 보상은 placeholder (Layer 5 결정)
Y_L2_PLACEHOLDER = Y_L3 * 2.0
Y_L2_L3_PLACEHOLDER = Y_L3 + Y_L2_PLACEHOLDER * 0.5

PAYOUT_PER_CASE = {
    'A': 0,
    'B': Y_L3,                        # ★ Part 4에서 도출된 실제값
    'C': Y_L2_PLACEHOLDER,            # placeholder
    'D': Y_L2_L3_PLACEHOLDER,         # placeholder
}

PAYOUT_STATUS = {
    'A': '보상 없음',
    'B': '✓ Part 4 도출',
    'C': '⚠ L5 결정 (내부 placeholder)',
    'D': '⚠ L5 결정 (내부 placeholder)',
}

CASE_NAMES = {
    'A': '정상 운영',
    'B': 'L3 단독 (지수형)',
    'C': 'L2 단독 (실손)',
    'D': 'L2 + L3 병합',
}

def count_case_events(case_series, target_case):
    is_case = (case_series == target_case).astype(int)
    starts = (is_case == 1) & (is_case.shift(1, fill_value=0) == 0)
    return int(starts.sum())

case_summary = []
for case in ['A', 'B', 'C', 'D']:
    n_minutes = int((layer3['case'] == case).sum())
    n_events = count_case_events(layer3['case'], case)
    payout = PAYOUT_PER_CASE[case]
    
    case_summary.append({
        'case': case,
        'name': CASE_NAMES[case],
        'n_minutes': n_minutes,
        'n_events': n_events,
        'percent_of_time': n_minutes / len(layer3) * 100,
        'payout_per_event_krw_internal': payout,   # 내부 계산용
        'total_payout_krw_internal': payout * n_events,   # 내부 계산용
        'payout_status': PAYOUT_STATUS[case],
    })

case_df = pd.DataFrame(case_summary)
case_df['payout_per_event_billion_internal'] = case_df['payout_per_event_krw_internal'] / 1e8
case_df['total_payout_billion_internal'] = case_df['total_payout_krw_internal'] / 1e8

print('=== 케이스별 시뮬 결과 (v3) ===')
display(case_df[['case', 'name', 'n_minutes', 'n_events', 'percent_of_time', 'payout_status']])

In [ ]:
# v3: Layer 3 책임 영역 명확히
case_b_events = case_df.loc[case_df['case'] == 'B', 'n_events'].iloc[0]
case_b_payout = case_df.loc[case_df['case'] == 'B', 'total_payout_billion_internal'].iloc[0]

print('=== Layer 3 책임 영역 (Part 4에서 정량화 완료) ===')
print(f'Case B (L3 단독): {case_b_events} events, 총 {case_b_payout:.2f}억 (데이터 기간)')
print()
print('=== Layer 5 결정 영역 ===')
print(f'Case C: {case_df.loc[case_df["case"] == "C", "n_events"].iloc[0]} events — 보상액 L5 결정')
print(f'Case D: {case_df.loc[case_df["case"] == "D", "n_events"].iloc[0]} events — 병합 보상 정책 L5 결정')

## Step E. Sensitivity 분석 (v3: 표현 완화)

**v3 변경**: "L2 독립적" → "L2 가정에 민감하지 않음"

In [ ]:
def simulate_with_accuracy(accuracy):
    """v3: L3_trigger 기준 시뮬"""
    np.random.seed(42)
    p_process = np.where(infra_actual, P_PROCESS_FAULT_GIVEN_INFRA, P_PROCESS_FAULT_GIVEN_NORMAL)
    pf = np.random.binomial(1, p_process)
    
    np.random.seed(43)
    p_ai = np.where(pf == 1, accuracy, 1 - accuracy)
    ai_dec = np.random.binomial(1, p_ai)
    ai_corr = (pf == ai_dec).astype(int)
    
    # ★ v3: L3_trigger 기준 분류
    case_series = pd.Series(
        np.where(
            (~infra_trigger) & (ai_corr == 1), 'A',
            np.where(
                (infra_trigger) & (ai_corr == 1), 'B',
                np.where(
                    (~infra_trigger) & (ai_corr == 0), 'C',
                    'D'
                )
            )
        ),
        index=layer3.index
    )
    
    result = {'L2_accuracy': accuracy}
    for case in ['A', 'B', 'C', 'D']:
        result[f'{case}_events'] = count_case_events(case_series, case)
    return result

accuracy_grid = [0.60, 0.70, 0.80, 0.90, 0.95]
sens_results = pd.DataFrame([simulate_with_accuracy(a) for a in accuracy_grid])

print('=== Sensitivity 분석 (v3: L3_trigger 기준) ===')
display(sens_results.round(2))

# Case B 변동성
b_variance = sens_results['B_events'].std() / max(sens_results['B_events'].mean(), 1)
print(f'\nCase B (L3 단독) 변동계수: {b_variance:.4f}')
if b_variance < 0.1:
    print('✓ L2 정확도 가정에 민감하지 않음 — L3 단독 보상 부담 안정적')
else:
    print(f'⚠️ L2 정확도에 다소 영향 받음 (변동계수 {b_variance:.2f})')

## Step F. 연간 환산 ⭐ v3: unique date 개수

**v2 버그**: `n_days = (max - min).days + 1` → 불연속 일자에서 과대 계산 → 연간 환산 과소 추정

**v3 수정**: `n_days = layer3.index.normalize().nunique()` → 실제 일자 수

In [ ]:
# v2 방식 (참고용)
n_days_v2 = (layer3.index.max() - layer3.index.min()).days + 1

# ★ v3: 실제 unique date 수
n_days = layer3.index.normalize().nunique()

DAYS_PER_YEAR = 365
scale_factor = DAYS_PER_YEAR / max(n_days, 1)
scale_factor_v2 = DAYS_PER_YEAR / max(n_days_v2, 1)

print(f'데이터 기간 비교:')
print(f'  v2 방식 (min~max): {n_days_v2}일 → scale_factor {scale_factor_v2:.1f}')
print(f'  v3 방식 (unique):  {n_days}일 → scale_factor {scale_factor:.1f}')
if n_days != n_days_v2:
    print(f'  ★ 차이 발견! v2는 {scale_factor_v2/scale_factor:.2f}배 과소 추정')

annual_summary = []
for case in ['A', 'B', 'C', 'D']:
    row = case_df[case_df['case'] == case].iloc[0]
    annual_events = row['n_events'] * scale_factor
    annual_summary.append({
        'case': case,
        'name': CASE_NAMES[case],
        'annual_events': annual_events,
        'payout_status': PAYOUT_STATUS[case],
    })
annual_df = pd.DataFrame(annual_summary)

# Layer 3 부담
case_b_annual_events = annual_df.loc[annual_df['case'] == 'B', 'annual_events'].iloc[0]
case_d_annual_events = annual_df.loc[annual_df['case'] == 'D', 'annual_events'].iloc[0]
L3_annual_payout_billion = case_b_annual_events * Y_L3 / 1e8
L3_total_annual_payout_billion = (case_b_annual_events + case_d_annual_events) * Y_L3 / 1e8

print(f'\n=== 연간 환산 (v3: {n_days}일 unique → 365일) ===')
display(annual_df)

print(f'\n★ Layer 3 보상 부담 (v3 정확) ★')
print(f'   Case B만: 연간 {case_b_annual_events:.1f}회, {L3_annual_payout_billion:.2f}억원')
print(f'   Case B+D (L3 책임 분): 연간 {case_b_annual_events + case_d_annual_events:.1f}회, {L3_total_annual_payout_billion:.2f}억원')

## Step G. 시각화 (v3: L3_trigger 강조)

In [ ]:
# 2x2 매트릭스 (v3: 축 레이블 변경)
fig, ax = plt.subplots(figsize=(11, 8))

cell_data = {
    (0, 0): ('A', CASE_NAMES['A'], '#E8F5E9', 'no payout'),
    (0, 1): ('C', CASE_NAMES['C'], '#FFF3E0', 'L5 결정'),
    (1, 0): ('B', CASE_NAMES['B'], '#E3F2FD', f'Y_L3 = {Y_L3/1e8:.2f}억'),
    (1, 1): ('D', CASE_NAMES['D'], '#FFEBEE', 'L5 결정'),
}

for (row, col), (case, name, color, payout_str) in cell_data.items():
    n_events = int(case_df[case_df['case'] == case]['n_events'].iloc[0])
    pct = case_df[case_df['case'] == case]['percent_of_time'].iloc[0]
    annual_ev = annual_df[annual_df['case'] == case]['annual_events'].iloc[0]
    
    edge_width = 4 if case == 'B' else 2
    edge_color = 'darkblue' if case == 'B' else 'black'
    
    rect = plt.Rectangle((col, -row-1), 1, 1, 
                         facecolor=color, edgecolor=edge_color, linewidth=edge_width)
    ax.add_patch(rect)
    
    ax.text(col + 0.5, -row - 0.15, f'Case {case}',
            ha='center', va='top', fontsize=18, fontweight='bold')
    ax.text(col + 0.5, -row - 0.32, name,
            ha='center', va='top', fontsize=11)
    ax.text(col + 0.5, -row - 0.50, f'{pct:.1f}% of time',
            ha='center', va='top', fontsize=10, color='gray')
    ax.text(col + 0.5, -row - 0.65, f'{n_events} events (연간 {annual_ev:.0f}회)',
            ha='center', va='top', fontsize=10, color='gray')
    
    color_text = 'darkblue' if case == 'B' else ('green' if case == 'A' else 'darkorange')
    weight = 'bold' if case == 'B' else 'normal'
    ax.text(col + 0.5, -row - 0.85, payout_str,
            ha='center', va='top', fontsize=11, color=color_text, fontweight=weight)

ax.text(0.5, 0.3, 'AI 정탐 성공', ha='center', va='center', fontsize=14, fontweight='bold')
ax.text(1.5, 0.3, 'AI 정탐 실패', ha='center', va='center', fontsize=14, fontweight='bold')
ax.text(-0.4, -0.5, 'L3 trigger\n미발동', ha='center', va='center', fontsize=13, fontweight='bold', rotation=90)
ax.text(-0.4, -1.5, 'L3 trigger\n발동', ha='center', va='center', fontsize=13, fontweight='bold', rotation=90)

ax.set_xlim(-0.7, 2.2)
ax.set_ylim(-2.3, 0.6)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('2x2 책임 매트릭스 (v3: L3 trigger 기준)', fontsize=15, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

In [ ]:
# Sensitivity 차트
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#4CAF50', '#2196F3', '#FF9800', '#F44336']
axes[0].bar(case_df['case'], case_df['n_events'], color=colors, alpha=0.7, edgecolor='black')
axes[0].set_xlabel('Case')
axes[0].set_ylabel('Event 수')
axes[0].set_title('케이스별 Event 빈도 (v3)')
for i, n in enumerate(case_df['n_events']):
    axes[0].text(i, n, str(int(n)), ha='center', va='bottom', fontweight='bold')
axes[0].patches[1].set_edgecolor('darkblue')
axes[0].patches[1].set_linewidth(3)

axes[1].plot(sens_results['L2_accuracy'], sens_results['B_events'],
             marker='o', markersize=10, linewidth=3, color='darkblue', label='Case B (L3 단독)')
axes[1].plot(sens_results['L2_accuracy'], sens_results['A_events'],
             marker='s', linewidth=1.5, color='green', alpha=0.6, label='Case A')
axes[1].plot(sens_results['L2_accuracy'], sens_results['C_events'],
             marker='^', linewidth=1.5, color='orange', alpha=0.6, label='Case C')
axes[1].plot(sens_results['L2_accuracy'], sens_results['D_events'],
             marker='d', linewidth=1.5, color='red', alpha=0.6, label='Case D')
axes[1].axvline(x=0.70, color='gray', linestyle='--', alpha=0.5, label='기본 가정 (70%)')
axes[1].set_xlabel('L2 정탐 정확도 가정')
axes[1].set_ylabel('Event 수')
axes[1].set_title('Sensitivity: Case B 안정성 검증')
axes[1].legend(loc='best')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print('★ 핵심 관찰 ★')
print(f'Case B 변동계수 {b_variance:.4f} — L2 가정에 민감하지 않음')

## Step H. 발표용 결과 표 (v3: C/D 금액 숨김)

In [ ]:
# v3: C/D는 금액 대신 "L5 결정" 표시
presentation_table = pd.DataFrame({
    'Case': ['A', 'B', 'C', 'D'],
    'L3 trigger': ['미발동', '발동', '미발동', '발동'],
    'AI 정탐': ['성공', '성공', '실패', '실패'],
    '보험 유형': ['보상 없음', 'L3 지수형', 'L2 실손', 'L2+L3 병합'],
    '연간 Events': [f'{annual_df.loc[i, "annual_events"]:.0f}' for i in range(4)],
    '담당': ['—', 'Layer 3 (본 연구)', 'Layer 5 결정', 'Layer 5 결정'],
    '약정금': ['—', f'{Y_L3/1e8:.2f}억/event', 'L5 결정', 'L5 결정'],
})
print('=== 발표용 결과 표 (v3: 책임 영역 명확) ===')
display(presentation_table)

In [ ]:
# 결론 멘트 (v3 표현 완화)
print('=== 발표 결론 멘트 (v3) ===\n')

print('1) Layer 3 명확한 보상 부담:')
print(f'   "Layer 3 단독 trigger (Case B)는 연간 {case_b_annual_events:.1f}회,')
print(f'    {L3_annual_payout_billion:.2f}억원 지급으로 정량화."')
print()
print('2) Sensitivity (표현 완화):')
print(f'   "L2 정확도 60~95% sensitivity 분석에서 Case B 변동계수 {b_variance:.3f}.')
print(f'    L2 가정에 과도하게 민감하지 않음을 확인."')
print()
print('3) 책임 분리:')
print(f'   "Case C, D 보상액은 본 연구 범위 밖. Layer 5 보험 상품 설계 단계에서 결정."')
print()
print('4) Synthetic 한계:')
print(f'   "AI 정탐은 synthetic scenario ({L2_DETECTION_ACCURACY:.0%} 보수). 실제 통합 시 교체."')
print()
print('5) Part 5 포지션:')
print(f'   "Part 5는 보험금 산정 결론이 아닌 Layer 2-3 결합 시 가능한 보상 케이스를 보여주는 scenario simulation."')

## Step I. 저장

In [ ]:
# 발표용 (C/D 금액 없는 버전)
case_df_publish = case_df[['case', 'name', 'n_minutes', 'n_events', 'percent_of_time', 'payout_status']].copy()
case_df_publish.to_csv(os.path.join(OUT, 'case_simulation_results.csv'), index=False)
print(f'저장: case_simulation_results.csv (발표용, C/D 금액 제거)')

# 내부 계산용 (placeholder 금액 포함)
case_df.to_csv(os.path.join(OUT, 'case_simulation_internal.csv'), index=False)
print(f'저장: case_simulation_internal.csv (내부 계산용)')

sens_results.to_csv(os.path.join(OUT, 'sensitivity_analysis.csv'), index=False)
print(f'저장: sensitivity_analysis.csv')

presentation_table.to_csv(os.path.join(OUT, 'presentation_table.csv'), index=False)
print(f'저장: presentation_table.csv')

layer3.to_parquet(os.path.join(OUT, 'layer3_with_l2_scenario.parquet'))
print(f'저장: layer3_with_l2_scenario.parquet')

# v3 metadata (synthetic 강조 추가)
p5_metadata = {
    'version': 'v3',
    'note_strong': (
        '★ THIS IS A SYNTHETIC SCENARIO SIMULATION, NOT AN ACTUAL PRICING RESULT. ★ '
        'Layer 2 detection results are generated by statistical assumption. '
        'L2 and L2+L3 payout amounts are internal placeholders for simulation only. '
        'Part 5 demonstrates the framework for L2-L3 integration, '
        'not the final insurance product pricing.'
    ),
    'phase5_summary': {
        'cases_v3': {
            'A': '정상 운영 (L3 trigger X + AI 정탐 OK)',
            'B': 'L3 단독 — Layer 3 책임 (trigger 발동 + AI 정탐 OK)',
            'C': 'L2 단독 — L5 결정 (trigger X + AI 오탐)',
            'D': 'L2+L3 병합 — L5 결정 (trigger 발동 + AI 오탐)',
        },
        'classification_basis': 'L3_trigger (NOT incident_flag) — 보험은 trigger 발동 기준',
        'annual_case_events': {
            row['case']: float(row['annual_events']) 
            for _, row in annual_df.iterrows()
        },
        'L3_only_annual_events_case_B': float(case_b_annual_events),
        'L3_only_annual_payout_billion_krw': float(L3_annual_payout_billion),
        'L3_robustness_coefficient_of_variation': float(b_variance),
        'L3_robustness_statement': (
            'L2 가정에 과도하게 민감하지 않음 (variation coefficient < 0.1)'
            if b_variance < 0.1
            else f'L2 정확도에 다소 영향 (variation coefficient {b_variance:.3f})'
        ),
    },
    'assumptions': {
        'L2_detection_accuracy': L2_DETECTION_ACCURACY,
        'L2_accuracy_note': '70% (보수). Sensitivity 60~95%로 robustness 확인.',
        'P_process_fault_given_infra': P_PROCESS_FAULT_GIVEN_INFRA,
        'P_process_fault_given_normal': P_PROCESS_FAULT_GIVEN_NORMAL,
        'Y_L3_krw': float(Y_L3),
        'Y_L2_placeholder_internal': 'Layer 5 결정 (시뮬용 내부값)',
        'Y_L2_L3_placeholder_internal': 'Layer 5 결정 (시뮬용 내부값)',
    },
    'sensitivity_grid': accuracy_grid,
    'data_period': {
        'unique_dates': int(n_days),
        'min_to_max_days': int(n_days_v2),
        'scale_factor': float(scale_factor),
        'note_v3': 'n_days is unique date count, not (max-min). Critical for accurate annual scaling.',
    },
    'v3_changes': {
        '1_case_classification': 'incident_flag → L3_trigger (보험 도메인 정합)',
        '2_n_days': '(max-min).days → unique date count (불연속 일자에서 정확)',
        '3_variable_naming': 'infra_actual / infra_trigger 분리',
        '4_sensitivity_wording': '"독립적" → "민감하지 않음" 완화',
        '5_publish_vs_internal': '발표용 case_df는 C/D 금액 제거, 내부용은 별도 저장',
        '6_metadata_synthetic': 'note_strong 추가로 "not pricing result" 명시',
    },
}
with open(os.path.join(OUT, 'phase5_metadata.json'), 'w', encoding='utf-8') as f:
    json.dump(p5_metadata, f, ensure_ascii=False, indent=2)
print(f'저장: phase5_metadata.json (v3 synthetic 강조)')

print('\n=== Part 5 v3 완료 ===')
print(f'★ Case B (L3 단독, v3 정확 계산):')
print(f'   연간 {case_b_annual_events:.1f} events × {Y_L3/1e8:.2f}억 = {L3_annual_payout_billion:.2f}억원')
print(f'★ Sensitivity: 변동계수 {b_variance:.4f} (60~95% 범위)')
print(f'★ Scale factor: {scale_factor:.1f} (v3 unique date 기준)')

## ✅ Part 5 v3 완료 체크리스트

### v2 → v3 변경 (6개)
- [x] **1. ★ 케이스 분류: `incident_flag` → `L3_trigger`** (보험 정합성)
- [x] **2. ★ `n_days` unique date 개수** (연간 환산 정확)
- [x] 3. `infra_actual` / `infra_trigger` 변수 분리
- [x] 4. "L2 독립적" → "민감하지 않음" 완화
- [x] 5. C/D placeholder 금액 발표용 표에서 제거 (내부용 별도)
- [x] 6. metadata `note_strong` synthetic 강조

## 출력 파일 6개

| 파일 | 용도 |
|---|---|
| `case_simulation_results.csv` | **발표용** (C/D 금액 없음) |
| `case_simulation_internal.csv` | 내부 계산용 (placeholder 포함) |
| `sensitivity_analysis.csv` | L2 정확도별 |
| `presentation_table.csv` | 발표 슬라이드용 |
| `layer3_with_l2_scenario.parquet` | Synthetic 시계열 |
| `phase5_metadata.json` | 가정 + 결과 (v3 강조) |

## 발표 방어 멘트 (v3 강화)

| 질문 | 답변 |
|---|---|
| "왜 incident_flag 아닌 L3_trigger?" | "보험은 trigger 발동에 따라 지급. 실제 장애여도 trigger 안 발동하면 보상 없음 — 보험 도메인 정합" |
| "n_days 계산?" | "실제 unique date 개수. min~max로 계산하면 불연속 일자에서 과대 — 연간 환산 정확성" |
| "Sensitivity 결과 해석?" | "L2 정확도 60~95%에서 Case B 변동계수 < 0.1 — L2 가정에 과도하게 민감하지 않음" |
| "C, D 금액?" | "L5 결정 사항. 본 표에는 의도적으로 금액 없음. 시뮬 내부값은 별도 파일" |
| "Part 5의 위치?" | "보험금 산정 결론이 아닌 L2-L3 결합 시나리오 시뮬레이션. 본 연구는 framework 입증" |